# V2-P2 — `p2_b`'s generation: 50 files with `e16_b5`'s decode and the piece rule

Registration: `research/v2/PREREG_V2_P2.md` §6. Run **only after the laptop has verified
`p2_b`'s training** (VERIFY_TRAIN). E12's 50 jobs at E12's seeds, decoded raw + D1 + D1′ + B5,
exactly as `e16_b5`, plus the piece rule: a rest piece only in the path, query and fragment, a
host piece only in the name and subdomains. Every file records the probability mass the rule
removed and how many pieces were written outside their fields (registered: none).

**Attach two inputs:** **`uie-v2p2-code`**, and the **V2P2_train_b** notebook's **output**.
**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 1.5–2.5 h.
A rerun keeps every finished file. When done, download `/kaggle/working/v2p2_gen_b`.

In [ ]:
ARM = "p2_b"

# --- 1. locate the code bundle (and the inputs) --------------------------------------
import gc, hashlib, json, math
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("V2P2_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-v2p2-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)


def output_of(arm):
    hits = sorted(p for p in root.rglob("PROVENANCE.json") if (p.parent / arm).is_dir()) if root.exists() else []
    return hits[0].parent if hits else None

SRC = output_of(ARM)
if SRC is None:
    raise SystemExit(f"Attach the V2P2_train_{ARM[-1]} notebook's output too, then rerun.")
tprov = json.loads((SRC / "PROVENANCE.json").read_text())
CKPT = SRC / ARM / "final.pt"
assert CKPT.is_file() and sha256(CKPT) == tprov["arms"][ARM]["final_sha256"], \
    f"{ARM}: final.pt is not the one V2P2_train recorded"
print("checkpoint verified:", CKPT)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "v2", CODE / "research" / "e17", CODE / "research" / "e16",
          CODE / "research" / "e15", CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e15_common",
                      "e16_common", "e17_common", "v2_mem", "v2_bpe", "v2_p1_common", "v2_p2_common",
                      "v2_p2_tokenizer", "v2_p2_train")]:
    del sys.modules[name]
import uie
import e12_common as K
import e15_common as E15
import e16_common as E16
import e17_common as E17
import v2_bpe
import v2_p2_common as C
import v2_p2_tokenizer as TK
import v2_p2_train as T
manifest = json.loads((CODE / C.CODE_MANIFEST).read_text())
for mod in (uie, K, E15, E16, E17, v2_bpe, C, TK, T):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
bad += [t for t, h in manifest["piece_tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / C.STREAM_FILE) != manifest["stream"]["sha256"] or manifest["stream"]["sha256"] != C.STREAM_SHA256:
    bad.append(C.STREAM_FILE)
if sha256(CODE / C.E16_CAP_RECORD) != manifest["e16_cap_record"]:
    bad.append(C.E16_CAP_RECORD)
if manifest["piece_tables_registered"] != C.TABLE_SHA256 or manifest["rules_sha256"] != C.rules_sha256():
    bad.append("the registered tables or rules")
if manifest["loss_constants"] != C.LOSS_CONSTANT_P2:
    bad.append("the registered C")
assert not bad, f"STOP. files differ from the manifest: {bad}"
PIECES = C.load_tables(ARM, CODE)            # raises unless each table is the registered one (V-T3)
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables, the piece tables {[h[:12] for h in C.TABLE_SHA256.values()]}, "
      f"stream 1 ({manifest['stream']['rows']:,} rows) and e16_cap's record verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
GPU = torch.cuda.get_device_name(0)
print("torch:", torch.__version__, "| gpu:", GPU)

In [ ]:
# --- 3. tables, the registered jobs, the model and the piece rule ---------------------
from uie.controlled import StateSuffixTables
from uie.suffixes import SuffixTable

shipped = SuffixTable.load(CODE / "suffix_table.json")
tables = StateSuffixTables.load(CODE / "state_suffix_tables.json")
presence_slash = json.loads((CODE / "presence_slash_table.json").read_text())["table"]
device = torch.device("cuda")
OUT = WORK / "v2p2_gen_b"
OUT.mkdir(parents=True, exist_ok=True)
jobs = sorted(C.jobs(ARM), key=lambda j: (0 if j["mode"] == "free" else 1, j["label"], j["state"] or "", j["g"]))
print(f"{len(jobs)} files, {sum(j['n'] for j in jobs):,} URLs; decode {C.decode_kwargs()} + the piece rule")
CKPT_SHA = {ARM: sha256(CKPT)}
model, tok, _ = C.load_checkpoint(CKPT, device=device)
model.to(device).eval()
assert all(p.device.type == "cuda" for p in model.parameters()), "STOP. the model is not on the GPU"
assert tok.presence_plan and tok.root_slash and tok.slash_plan and tok.n_base == 110
assert {k: C.table_sha256(v) for k, v in tok._table_state.items()} == {k: C.TABLE_SHA256[k] for k in C.ARMS[ARM]}, \
    "STOP. the checkpoint's tables are not the registered ones"
assert model.cfg.d_model == E16.D_MODEL and sum(p.numel() for p in model.parameters()) == C.params(tok)
rule = C.PieceRule(model, tok).to(device).eval()
print(f"loaded {ARM}: width {model.cfg.d_model}, vocab {tok.vocab_size:,} "
      f"({ {k: len(tok.ids_of_table(k)) for k in ('rest', 'host')} } pieces), max_len {model.cfg.max_len}; "
      f"the piece rule: {tok.field_tables()}")

In [ ]:
# --- 4. generate -----------------------------------------------------------------------
# PROVENANCE.json is rewritten after EVERY file; a rerun keeps a file only with a
# matching record -- anything else is made again: same seed, same bytes.
import platform
import time

PROV = OUT / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"files": {}}
prov.update({"purpose": f"V2-P2 (research/v2/PREREG_V2_P2.md): {ARM}, decoded as e16_b5 plus the piece rule",
             "checkpoint_file": "final.pt", "code_commit": manifest["commit"],
             "checkpoints": CKPT_SHA, "tables": manifest["tables"], "piece_tables": C.TABLE_SHA256,
             "rules_sha256": C.rules_sha256(), "decode": C.decode_kwargs(),
             "environment": {"torch": torch.__version__, "python": platform.python_version(),
                             "gpu": GPU, "tldextract": metadata.version("tldextract")}})
t_start = time.time()
done = 0
for j in jobs:
    path = OUT / j["file"]
    rec = prov["files"].get(j["file"])
    if rec and path.is_file() and sha256(path) == rec["sha256"]:
        done += 1
        continue
    t0 = time.time()
    trace, qtrace, grec = [], [], {}
    before = rule.snapshot()
    if j["mode"] == "free":
        urls = C.free_urls(model, tok, j["label"], j["g"], shipped, presence_slash, device=device,
                           plan_trace=trace, query_trace=qtrace, rule=rule, record=grec)
    else:
        urls = C.control_urls(model, tok, j["label"], j["state"], j["g"], tables, presence_slash,
                              device=device, plan_trace=trace, query_trace=qtrace, rule=rule, record=grec)
    assert len(urls) == j["n"], (j["file"], len(urls))
    piece, rules = rule.record(since=before), grec["token_rules"]
    path.write_text(json.dumps(urls))
    prov["files"][j["file"]] = {**{k: v for k, v in j.items() if k != "file"},
                                "sha256": sha256(path), "seconds": round(time.time() - t0, 1),
                                "plan_trace_mean": sum(trace) / len(trace) if trace else None,
                                "query_moved_mean": sum(qtrace) / len(qtrace) if qtrace else None,
                                "query_rows_moved": sum(x > 0 for x in qtrace) if qtrace else None,
                                "slash_moved_mean": grec["slash_moved_mean"],
                                "slash_rows_moved": grec["slash_rows_moved"],
                                "piece_rule": piece, "token_rules": rules,
                                "cut_at_max_len": grec["cut_at_max_len"]}
    PROV.write_text(json.dumps(prov, indent=1))
    done += 1
    el = (time.time() - t_start) / 60
    print(f"  {done:>3}/{len(jobs)}  {j['file']:<46} {time.time() - t0:6.0f}s  ({el:.0f} min so far)"
          f"  violations: pieces {piece['violations']}, D1 {rules['d1_violations']}, B5 {rules['b5_violations']}",
          flush=True)
bad = {f: (r["piece_rule"]["violations"], r["token_rules"]["d1_violations"], r["token_rules"]["b5_violations"])
       for f, r in prov["files"].items()
       if r["piece_rule"]["violations"] or r["token_rules"]["d1_violations"] or r["token_rules"]["b5_violations"]}
print(f"\nall {len(jobs)} files written; violations (pieces, D1, B5): {bad or 'none'}. Now download {OUT}.")